In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim, StringType

#Reading From Silver Table

In [0]:
%sql 
SELECT * FROM silver.inventory_snapshot

In [0]:
query = """
            SELECT
                S.store_key, 
                IS.store_id,
                P.product_key,
                IS.product_id,
                D.date_key,
                IS.snapshot_date,
                IS.units_on_hand
            FROM silver.inventory_snapshot IS
            LEFT JOIN gold.dim_product P
            ON IS.product_id = P.product_id
            LEFT JOIN gold.dim_store S
            ON IS.store_id = S.store_id
            LEFT JOIN gold.dim_date D
            ON IS.snapshot_date = D.full_day"""
df = spark.sql(query)

In [0]:
df.display()

#Write into gold layer

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.fact_inventory_snapshot")
)

In [0]:
%sql
SELECT * FROM gold.fact_inventory_snapshot